In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%sql
DROP TABLE IF EXISTS bronze_dev.redfin.housing_market_zipcode;
DROP TABLE IF EXISTS bronze_dev.redfin.housing_market_county;

DROP TABLE IF EXISTS bronze_dev.redfin.delistings_relistings_neighborhood;
DROP TABLE IF EXISTS bronze_dev.redfin.delistings_relistings_zipcode;
DROP TABLE IF EXISTS bronze_dev.redfin.delistings_relistings_county;


DROP TABLE IF EXISTS bronze_dev.redfin.price_drops_county;
DROP TABLE IF EXISTS bronze_dev.redfin.price_drops_county_zipcode;
DROP TABLE IF EXISTS bronze_dev.redfin.price_drops_neighborhood;
DROP TABLE IF EXISTS bronze_dev.redfin.price_drops_zipcode;


In [0]:
%sql
SELECT * FROM bronze_dev.opportunity_insights.social_capital_zip;

In [0]:
from datetime import datetime
from src.utils.sources_ref import REDFIN_TABLES, REDFIN_VOLUME_BASE, OPPORTUNITY_INSIGHTS_SOCIAL_CAPITAL_CONFIG, CENSUS_BUREAU_AMERICAN_COMMUNITY_SURVEY_CONFIG

import json
import requests

### Redfin

In [0]:

def ingest_redfin_tables():
    for t in REDFIN_TABLES:
        
        # Get the redfin directory (eg, "deslistings_relisting") and grain (county, zip, neighborhood)
        d = t["redfin_dir"]
        g = t["redfin_grain"]
        table_name = t["table_name"]

        src = f"s3://redfin-public-data/redfin_data_center/{d}/monthly/all_{g}.csv"
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest = f"{REDFIN_VOLUME_BASE}/{table_name}/{d}_monthly_all_{g}_{timestamp}.csv"


        dbutils.fs.cp(
            src,
            dest
        )
        print(f"Success: {src} -> {dest}")

ingest_redfin_tables()

### Opportunity Insights - Social Capital Data

In [0]:

def ingest_opportunity_social_capital_table():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dir = OPPORTUNITY_INSIGHTS_SOCIAL_CAPITAL_CONFIG["csv_directory"]
    dest = f"{dir}/social_capital_zip_{timestamp}.csv"
    src = (
        "https://data.humdata.org/dataset/85ee8e10-0c66-4635-b997-79b6fad44c71/"
        "resource/ab878625-279b-4bef-a2b3-c132168d536e/download/social_capital_zip.csv"
    )

    dbutils.fs.cp(
        src,
        dest
    )
    print(f"Success: {src} -> {dest}")

ingest_opportunity_social_capital_table()

### Census Bureau American Community Survey Data

In [0]:


def fetch_census_bureau_acs():
    """
    Download the American Community Survey ZCTA-level data from the Census Bureau
    """
    API_KEY = dbutils.secrets.get(scope="api-secrets", key="census-api-key" ).lstrip("\x00")

    src = "https://api.census.gov/data/2023/acs/acs5"

    column_list = [
            "NAME",

            # Total population
            "B01003_001E",

            # Median age
            "B01002_001E",

            # Median household income
            "B19013_001E",

            # Poverty count
            "B17001_002E",

            # Race counts
            "B02001_002E",  # White alone
            "B02001_003E",  # Black alone
            "B02001_005E",  # Asian alone

            # Hispanic population
            "B03003_003E"
        ]

    params = {
        "get": ",".join(column_list),
        "for": "zip code tabulation area:*",
        "key": API_KEY
    }

    response = requests.get(src, params=params)
    data = response.json()

    csv_dir = CENSUS_BUREAU_AMERICAN_COMMUNITY_SURVEY_CONFIG["csv_directory"]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dest = f"{csv_dir}/social_capital_zip_{timestamp}.json"

    with open(dest, "w") as f:
        json.dump(data, f, indent=2, default=str)

    print(f"Success: {src} -> {dest}")


    from pyspark.sql.functions import lit
    from pyspark.sql import DataFrame
    # Extract header and rows
    columns = data[0]
    rows = data[1:]

    # Create Spark DataFrame directly
    df = spark.createDataFrame(rows, schema=columns)

fetch_census_bureau_acs()
